In [1]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

In [2]:
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import larch as lx
import numpy as np
import pandas as pd
from larch import PX

sys.path.insert(0, os.path.abspath(".."))
from lib import model_spec as lm
from lib import modeling_util as lut
from lib import io as lio


In [3]:
num_alternatives = 50
unixtime = int(datetime.now(UTC).timestamp())
path = "../data/estdata_10_2018_50.parquet"
data_file = Path(path).stem

### Read data

In [4]:
df_train = lio.read_estdata(
    path,
    num_alternatives,
)
print(df_train.shape)

(253090, 1062)


/workspace/migration/lib/io.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["person_id"] = np.arange(len(df))
/workspace/migration/lib/io.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ALT_CHOICE"] = 0


In [5]:
# need to have the sentinel values be different so that SAME_CBSA works correctly
df_train["NAME_NUM.ORIG"].min(), df_train["ALT1_CBSA"].min()

(np.int64(-2), np.float32(-1.0))

### Look at collinearity in requested columns

In [6]:
# stack all alternatives into long format
frames = []
for i in range(1, num_alternatives + 1):
    cols = {
        f"ALT{i}_{v}": v
        for v in lm.required_alt_suffixes()
        if f"ALT{i}_{v}" in df_train.columns
    }
    frames.append(df_train[list(cols)].rename(columns=cols))

long = pd.concat(frames, ignore_index=True)

# add the transformed versions you actually use in the model
long["log_DIST"] = np.log(long["DIST"] + 1)
long["log_TOT_POP"] = np.log(long["TOT_POP"])

corr = long.corr()

In [7]:
c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.3])

TOT_POP                   log_TOT_POP           0.992355
DIST                      log_DIST              0.881614
MED_RENT_K                MED_EARNINGS_K        0.721018
TYPE                      MED_TRAVEL_TIME       0.611333
FOREIGN_BORN_PROP         MED_RENT_K            0.600982
TYPE                      MED_RENT_K            0.589543
HOUSE_VACANCY_PROP        LF_PARTCP_RATE        0.568650
MED_TRAVEL_TIME           MED_RENT_K            0.553458
FOREIGN_BORN_PROP         MED_TRAVEL_TIME       0.541512
                          TYPE                  0.528617
LF_PARTCP_RATE            MED_EARNINGS_K        0.503493
JAN_AVG_TEMP_C            CBSA                  0.498383
TYPE                      HOUSE_VACANCY_PROP    0.493443
                          LF_PARTCP_RATE        0.478337
MED_RENT_K                LF_PARTCP_RATE        0.472684
FOREIGN_BORN_PROP         JAN_AVG_TEMP_C        0.425443
HOUSE_VACANCY_PROP        HH_WITH_CHILD_PROP    0.407723
                          MED_R

In [8]:
orig_vars = [v for v in lm.required_individual_columns() if v in df_train.columns]  # skip any missing
corr = df_train[orig_vars].corr()

c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.3].to_string())

CHOSEN                                                ORIGIN_STATE                                      0.974723
FOREIGN                                               POBP                                              0.903096
Proportion of people in college.ORIG                  Proportion of people 18-34.ORIG                   0.800195
Proportion of people AAPI.ORIG                        Median gross rent in thousands of dollars.ORIG    0.792994
EDU_HIGH                                              EDU_BACHELORS                                     0.786110
Proportion foreign born.ORIG                          Median gross rent in thousands of dollars.ORIG    0.783520
Proportion of people Latino.ORIG                      Proportion of people White.ORIG                   0.783313
Proportion foreign born.ORIG                          Proportion of people White.ORIG                   0.777155
Median earnings in thousands of dollars.ORIG          Median gross rent in thousands of dollars.

### Reshape to long format, build the Larch dataset

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_torch_choice.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. 

The returned `long_df` is already sorted by `(person_id, alt)`; `Dataset.construct.from_idca` takes it
directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the torch-choice
port.


In [9]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

long_df.set_index(["person_id", "alt"], inplace=True)

In [10]:
lut.print_utility_formula(long_df.reset_index()[["person_id", "alt", "choice", "log_pop_offset"] + varnames])

Stay utility:
    Beta(stay)*Variable(stay)
    + Beta(stay_age_50_64)*Variable(stay_age_50_64)
    + Beta(stay_edu_high)*Variable(stay_edu_high)
    + Beta(stay_T34)*Variable(stay_T34)
    + Beta(log_pop_offset)*Variable(log_pop_offset)
    + Beta(stay_age_18_22)*Variable(stay_age_18_22)
    + Beta(stay_age_23_29)*Variable(stay_age_23_29)
    + Beta(stay_age_30_39)*Variable(stay_age_30_39)
    + Beta(stay_age_40_49)*Variable(stay_age_40_49)
    + Beta(stay_child_under_6)*Variable(stay_child_under_6)
    + Beta(stay_child_6_to_17)*Variable(stay_child_6_to_17)
    + Beta(stay_married_more_than_year)*Variable(stay_married_more_than_year)
    + Beta(stay_married_less_than_year)*Variable(stay_married_less_than_year)
    + Beta(stay_recently_divorced_or_widowed)*Variable(stay_recently_divorced_or_widowed)
    + Beta(stay_2work_mar)*Variable(stay_2work_mar)
    + Beta(stay_single_parent)*Variable(stay_single_parent)
    + Beta(stay_edu_college)*Variable(stay_edu_college)
    + Beta(stay_in_c

In [11]:
ds = lx.Dataset.construct.from_idca(
    long_df[["choice", "log_pop_offset"] + varnames], crack=False
)
ds

<xarray.Dataset> Size: 3GB
Dimensions:                                       (person_id: 253090, alt: 51)
Coordinates:
  * person_id                                     (person_id) int64 2MB 0 ......
  * alt                                           (alt) int64 408B 0 1 ... 49 50
Data variables: (12/55)
    choice                                        (person_id, alt) int64 103MB ...
    log_pop_offset                                (person_id, alt) float32 52MB ...
    stay                                          (person_id, alt) float32 52MB ...
    stay_age_18_22                                (person_id, alt) float32 52MB ...
    stay_age_23_29                                (person_id, alt) float32 52MB ...
    stay_age_30_39                                (person_id, alt) float32 52MB ...
    ...                                            ...
    destchoice_samecbsa                           (person_id, alt) float32 52MB ...
    destchoice_samestate                          (person_id, alt) float32 52MB ...
    destchoice_birthstate                         (person_id, alt) float32 52MB ...
    destchoice_T34                                (person_id, alt) float32 52MB ...
    destchoice_metro                              (person_id, alt) float32 52MB ...
    destchoice_same_cbsa_type                     (person_id, alt) float32 52MB ...
Attributes:
    _caseid_:  person_id
    _altid_:   alt

### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [12]:
m = lx.Model(ds)
m.title = f"us_mnl_{data_file}_{unixtime}"
m.compute_engine = "numba"

# all alternatives have the same utility function
# stay-specific columns have their values zeroed out for destination alternatives and vice versa
# PX represents a column multiplied by a coefficient that will be estimated
total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
# all alternatives are available for everyone
# no availability_ca_var needed.

# fix the size term coefficient, it is a constant
m.lock_value("log_pop_offset", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


In [13]:
# NOTE: nesting showed that mu_move tended to go towards 1, indicating that nesting is not necessary

# # optional cell: turns on the nested structure

# # nested logit: alt=0 (stay) stays a direct root child (== a degenerate nest fixed at 1.0);
# # alts 1..num_alternatives go under a "Move" nest with an estimated logsum coefficient.
# m.graph.new_node(
#     parameter="mu_move",
#     children=list(range(1, num_alternatives + 1)),
#     name="Move",
# )
# m.set_value("mu_move", value=0.5, initvalue=0.5, minimum=0.001, maximum=1.0)

# m.ordering = [
#     ("Stay", "stay.*"),
#     ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
#     ("Destination-only", "destchoice.*"),
#     ("Nesting", "mu_.*"),
#     ("Offset", "log_pop_offset"),
# ]
# m.title = f"us_nested_{data_file}_{unixtime}"


### Fitting

In [14]:
print("null log-likelihood:", m.loglike())


null log-likelihood: -726809.5990740261


In [15]:
result = m.maximize_loglike(method="BHHH")
result


┣          loglike: np.float64(-117928.39278523313)
┣                x: amenities_est_per_capita                       -81.880691
┃                   amenities_est_per_capita_18_34                  38.212542
┃                   amenities_est_per_capita_35_64                  17.110217
┃                   destchoice_T34                                  -0.456339
┃                   destchoice_birthstate                            0.305154
┃                   destchoice_logdist                              -0.950191
┃                   destchoice_metro                                -0.174610
┃                   destchoice_same_cbsa_type                        0.278641
┃                   destchoice_samecbsa                              1.505163
┃                   destchoice_samestate                             2.061510
┃                   jan_avg_temp_c                                   0.015385
┃                   lf_prop_if_in_lf                                -0.760114
┃                   log_pop_offset                                   1.000000
┃                   med_earnings_10k                                 0.067148
┃                   med_rent_1k                                     -0.578940
┃                   median_travel_time                              -0.032735
┃                   proportion_also_latino                           0.828996
┃                   proportion_also_mil                             26.055662
┃                   proportion_college_if_in_college                10.789892
┃                   proportion_foreign_if_foreign                    1.654392
┃                   proportion_hh_with_children_if_have_children     3.838729
┃                   proportion_same_age_18_34                        3.016558
┃                   proportion_same_age_35_64                        2.355395
┃                   proportion_same_age_65_plus                      3.543223
┃                   proportion_same_naics_agr_ext                    6.335788
┃                   proportion_same_naics_goods_trade                1.547612
┃                   proportion_same_naics_govt                       2.476873
┃                   proportion_same_naics_high_ed                    0.646308
┃                   proportion_same_race_aapi                        3.714500
┃                   proportion_same_race_black                       2.037423
┃                   proportion_same_race_indian                      4.701067
┃                   proportion_same_race_white                       2.194642
┃                   stay                                            -6.536192
┃                   stay_2work_mar                                   0.727179
┃                   stay_T34                                         0.236046
┃                   stay_age_18_22                                  -1.849304
┃                   stay_age_23_29                                  -1.784312
┃                   stay_age_30_39                                  -1.409256
┃                   stay_age_40_49                                  -0.964992
┃                   stay_age_50_64                                  -0.437537
┃                   stay_child_6_to_17                               0.416265
┃                   stay_child_under_6                              -0.213469
┃                   stay_edu_college                                -0.104462
┃                   stay_edu_high                                    0.054561
┃                   stay_foreign                                    -0.257257
┃                   stay_in_college                                 -0.109353
┃                   stay_married_less_than_year                     -0.712991
┃                   stay_married_more_than_year                      0.496335
┃                   stay_metro                                       0.140191
┃                   stay_mil                                        -1.513694
┃                   stay_naics_govt             

In [16]:
m.calculate_parameter_covariance()
m.parameter_summary()


/tmp/ipykernel_9630/1976790631.py:1: PossibleOverspecification: Model is possibly over-specified (hessian is nearly singular).
  m.calculate_parameter_covariance()


In [17]:
report = lx.Reporter(title=m.title)
report << "# Parameter Summary" << m.parameter_summary()
report << "# Estimation Statistics" << m.estimation_statistics()
report.save(
    f"results/{m.title}.html",
    overwrite=True,
    metadata=m.dumps(),
)
m.save(f"results/{m.title}_spec.yaml")